In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3

sns.set_theme(style="whitegrid")
print("Setup OK, pandas version:", pd.__version__)

Setup OK, pandas version: 3.0.6


In [3]:
df = pd.read_csv("../data/life_expectancy.csv")
df_raw = df.copy()
df.head()

,Country,Year,Status,Life expectancy,Adult Mortality,infant deaths,Alcohol,percentage expenditure,Hepatitis B,Measles,...,Polio,Total expenditure,Diphtheria,HIV/AIDS,GDP,Population,thinness 1-19 years,thinness 5-9 years,Income composition of resources,Schooling
0,Afghanistan,2015,Developing,65.0,263.0,62,0.01,71.279624,65.0,1154,...,6.0,8.16,65.0,0.1,584.259210,33736494.0,17.2,17.3,0.479,10.1
1,Afghanistan,2014,Developing,59.9,271.0,64,0.01,73.523582,62.0,492,...,58.0,8.18,62.0,0.1,612.696514,327582.0,17.5,17.5,0.476,10.0
2,Afghanistan,2013,Developing,59.9,268.0,66,0.01,73.219243,64.0,430,...,62.0,8.13,64.0,0.1,631.744976,31731688.0,17.7,17.7,0.470,9.9
3,Afghanistan,2012,Developing,59.5,272.0,69,0.01,78.184215,67.0,2787,...,67.0,8.52,67.0,0.1,669.959000,3696958.0,17.9,18.0,0.463,9.8
4,Afghanistan,2011,Developing,59.2,275.0,71,0.01,7.097109,68.0,3013,...,68.0,7.87,68.0,0.1,63.537231,2978599.0,18.2,18.2,0.454,9.5


In [4]:
df.shape

(2938, 22)

### Dataset Overview
The dataset contains **2,938 rows** and **22 columns**. Each row represents one country in one year, with columns for life expectancy, mortality, immunization coverage (Hepatitis B, Polio, Diphtheria), economic indicators (GDP, expenditure) and social indicators (schooling, income composition of resources).

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2938 entries, 0 to 2937
Data columns (total 22 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Country                          2938 non-null   str    
 1   Year                             2938 non-null   int64  
 2   Status                           2938 non-null   str    
 3   Life expectancy                  2928 non-null   float64
 4   Adult Mortality                  2928 non-null   float64
 5   infant deaths                    2938 non-null   int64  
 6   Alcohol                          2744 non-null   float64
 7   percentage expenditure           2938 non-null   float64
 8   Hepatitis B                      2385 non-null   float64
 9   Measles                          2938 non-null   int64  
 10   BMI                             2904 non-null   float64
 11  under-five deaths                2938 non-null   int64  
 12  Polio                          

### Data Types
`df.info()` shows that `Country` and `Status` are text columns, `Year`, `infant deaths`, `Measles` and `under-five deaths` are integers, and all other columns are `float64`. No numeric column is stored as text, so no type conversion is needed except for confirming `Year` as an integer.

Many columns have fewer non-null values than the 2,938 rows, which means they contain missing data. The column names are also inconsistent: some have leading or trailing spaces (for example ` BMI`, ` HIV/AIDS`, ` thinness  1-19 years`) and others contain spaces or special characters. These will be standardized before analysis.

In [6]:
(df.isnull().sum() / len(df) * 100).round(2).sort_values(ascending=False)

Population                         22.19
Hepatitis B                        18.82
GDP                                15.25
Total expenditure                   7.69
Alcohol                             6.60
Income composition of resources     5.68
Schooling                           5.55
 thinness  1-19 years               1.16
 thinness 5-9 years                 1.16
 BMI                                1.16
Diphtheria                          0.65
Polio                               0.65
Life expectancy                     0.34
Adult Mortality                     0.34
infant deaths                       0.00
Status                              0.00
Country                             0.00
Year                                0.00
under-five deaths                   0.00
Measles                             0.00
percentage expenditure              0.00
 HIV/AIDS                           0.00
dtype: float64

### Missing Values
The columns with the most missing data are:
- **Population** (22.19%, about 652 rows)
- **Hepatitis B** (18.82%, about 553 rows)
- **GDP** (15.25%, about 448 rows)

Moderate missingness appears in `Total expenditure` (7.69%), `Alcohol` (6.60%), `Income composition of resources` (5.68%) and `Schooling` (5.55%). `thinness 1-19 years`, `thinness 5-9 years` and `BMI` are missing about 1.16%, `Diphtheria` and `Polio` about 0.65%, and `Life expectancy` and `Adult Mortality` about 0.34% (10 rows each). The remaining columns have no missing values.

Because no column is missing more than about 22% of its values, dropping every incomplete row would discard a large part of the data. I will therefore handle each column according to its percentage and importance instead.

In [7]:
df.duplicated().sum()

np.int64(0)

### Duplicate Rows
No exact duplicate rows were found (`df.duplicated().sum()` returned 0), so no rows need to be removed for duplication.

### Summary of Initial Inspection
- The dataset has 2,938 rows and 22 columns covering multiple countries and years.
- Missing values are concentrated in `Population` (22.19%), `Hepatitis B` (18.82%) and `GDP` (15.25%), with smaller gaps in several other columns.
- `Life expectancy`, the main variable of interest, has only 10 missing rows (0.34%).
- Column names are inconsistent (extra spaces and special characters) and will be standardized.
- There are no exact duplicate rows.

**Cleaning plan:** standardize column names and country names, drop the few rows with missing life expectancy, impute the other numeric columns using each country's median (with the overall median as a fallback), and check outliers before saving the cleaned dataset.